# **From Waveform to Genre — Evaluation**

## Deep Learning Genre Classification

This notebook provides a critical evaluation of the deep learning pipeline developed in `data_prep.ipynb`, `models_train.ipynb`, and `analysis.ipynb` (`notebooks/deep_learning/`). It does not modify or re-run the original training code; it loads the already-generated results and metadata from Google Drive and focuses on questions that were not addressed in the original notebooks: sample size adequacy, data leakage risk, baseline comparisons, and a clearer interpretation of per-class performance and model behavior.

**FMA dataset (citation)**
Defferrard, M., Benzi, K., Vandergheynst, P., & Bresson, X. (2017). *FMA: A Dataset for Music Analysis*. 18th ISMIR. PDF: https://arxiv.org/pdf/1612.01840.pdf

**License / data note:** FMA metadata is licensed under CC BY 4.0; audio files are distributed under per-artist Creative Commons terms. This notebook uses only precomputed metadata, metrics, and evaluation results, and does not redistribute original audio files. Any use of FMA-derived materials should include proper attribution according to original dataset terms.

**Relationship to previous notebooks:** All results referenced here (`inference_results.csv`, `classification_report.json`, `confusion_matrix.png`, `training_history.csv`, `all_misclassifications.csv`, `train.csv`, `test.csv`) are outputs generated by the previous deep learning pipeline (`data_prep.ipynb`, `models_train.ipynb`, and `analysis.ipynb`) and stored in Google Drive under `waveform_analysis_outputs/data_storage`. No model retraining is performed in this notebook.  

**Execution order** Before running this notebook, data_prep.ipynb, models_train.ipynb, and analysis.ipynb must be executed in this order to generate the required datasets and evaluation artifacts. This notebook does not retrain the model; it only loads and analyzes the previously generated results.

## 1. Environment Setup & Artifact Loading

This section connects to Google Drive and loads all precomputed artifacts generated by the deep learning pipeline:
- `train.csv` & `test.csv` — Dataset splits and metadata
- `inference_results.csv` — Test set predictions and ground truth
- `classification_report.json` — Per-class precision, recall, and F1-score
- `training_history.csv` — Loss and accuracy progression per epoch
- `all_misclassifications.csv` — Detailed breakdown of incorrect predictions

In [ ]:
from google.colab import drive
# mount Google Drive to access files stored in it
drive.mount('/content/drive')

In [ ]:
import os
import shutil
import json
import pandas as pd

# path to the original files in Google Drive
DRIVE_SOURCE = '/content/drive/MyDrive/waveform_analysis_outputs/data_storage'

# local working directory in Colab (isolated from Drive)
LOCAL_STORAGE = '/content/evaluation_data'
os.makedirs(LOCAL_STORAGE, exist_ok=True)

# copying the files to Colab
if os.path.exists(DRIVE_SOURCE):
    for filename in os.listdir(DRIVE_SOURCE):
        src_file = os.path.join(DRIVE_SOURCE, filename)
        dst_file = os.path.join(LOCAL_STORAGE, filename)
        if os.path.isfile(src_file):
            shutil.copy2(src_file, dst_file)
            print(f"Copied: {filename}")
else:
    raise FileNotFoundError(f"Directory not found: {DRIVE_SOURCE}")

# the files are loaded into memory only from the local copy in Colab.
train_df = pd.read_csv(os.path.join(LOCAL_STORAGE, 'train.csv'))
test_df = pd.read_csv(os.path.join(LOCAL_STORAGE, 'test.csv'))
inference_df = pd.read_csv(os.path.join(LOCAL_STORAGE, 'inference_results.csv'))
history_df = pd.read_csv(os.path.join(LOCAL_STORAGE, 'training_history.csv'))

with open(os.path.join(LOCAL_STORAGE, 'classification_report.json'), 'r') as f:
    class_report = json.load(f)

print("-" * 50)
print(f"Train samples: {len(train_df)}")
print(f"Test samples:  {len(test_df)}")
print("All data is loaded from the local copy in Colab. The originals in Drive are intact.")

In [ ]:
# Check the exact size and class distribution of all dataset splits

split_dfs = {
    "train": train_df,
    "validation": pd.read_csv(os.path.join(LOCAL_STORAGE, "val.csv")),
    "test": test_df
}

for split_name, df in split_dfs.items():
    print(f"{split_name.capitalize()} samples: {len(df)}")
    print("Columns:", list(df.columns))
    print()

total_samples = sum(len(df) for df in split_dfs.values())
print(f"Total samples across all splits: {total_samples}")

## 1.1. Dataset Size and Structure

The dataset contains 800 audio tracks. They are divided into three subsets:

- **Training set** — 559 tracks
- **Validation set** — 121 tracks
- **Test set** — 120 tracks

This corresponds to an approximate split of 70%, 15%, and 15%, respectively.

This is a relatively small dataset for training a deep neural network. The limited size is particularly important because the task involves several music genres, and tracks within the same genre can vary considerably.

The split files contain only the track identifier (`track_id`) and its genre (`genre`). Therefore, it is possible to check whether the same track appears more than once or is present in more than one subset. However, it is not possible to determine whether tracks by the same artist or from the same album appear in both the training and test sets.

The lack of information about artists and albums is identified as a limitation of the original experiment.

In [ ]:
# Dataset size, class distribution, duplicate IDs, and split overlap

import pandas as pd

# аdd split labels without modifying the original DataFrames
train_audit = train_df.copy()
train_audit["split"] = "train"

validation_audit = split_dfs["validation"].copy()
validation_audit["split"] = "validation"

test_audit = test_df.copy()
test_audit["split"] = "test"

all_splits_df = pd.concat(
    [train_audit, validation_audit, test_audit],
    ignore_index=True
)

print("Dataset sizes")
print("=" * 40)
print(all_splits_df["split"].value_counts().reindex(
    ["train", "validation", "test"]
))
print(f"\nTotal rows: {len(all_splits_df)}")

print("\nClass distribution by split")
print("=" * 40)
class_distribution = pd.crosstab(
    all_splits_df["genre"],
    all_splits_df["split"]
).reindex(columns=["train", "validation", "test"], fill_value=0)

print(class_distribution)

print("\nDuplicate track IDs within each split")
print("=" * 40)
for split_name, split_df in [
    ("train", train_df),
    ("validation", split_dfs["validation"]),
    ("test", test_df)
]:
    duplicate_count = split_df["track_id"].duplicated().sum()
    print(f"{split_name}: {duplicate_count} duplicate IDs")

print("\nTrack ID overlap between splits")
print("=" * 40)

train_ids = set(train_df["track_id"])
validation_ids = set(split_dfs["validation"]["track_id"])
test_ids = set(test_df["track_id"])

print(f"Train ∩ Validation: {len(train_ids & validation_ids)}")
print(f"Train ∩ Test:       {len(train_ids & test_ids)}")
print(f"Validation ∩ Test:  {len(validation_ids & test_ids)}")

print("\nUnique track IDs in the complete dataset")
print("=" * 40)
print(f"Rows:              {len(all_splits_df)}")
print(f"Unique track IDs:  {all_splits_df['track_id'].nunique()}")
print(
    "All rows have unique track IDs:",
    len(all_splits_df) == all_splits_df["track_id"].nunique()
)

In [ ]:
import os

# check in local working folder (copied from Drive)
print("Files in LOCAL_STORAGE (/content/evaluation_data):")
print(sorted(os.listdir(LOCAL_STORAGE)))

# checking in the original Drive directory (data_storage)
print("\nFiles in Drive source (data_storage):")
print(sorted(os.listdir(DRIVE_SOURCE)))

# search for files associated with 'artist' or 'track' metadata (broader FMA dump)
keywords = ['track', 'artist', 'meta', 'fma']

print("\nSearching for potential metadata files by keyword match:")
for base_path, label in [(LOCAL_STORAGE, "local"), (DRIVE_SOURCE, "drive")]:
    for filename in os.listdir(base_path):
        if any(k in filename.lower() for k in keywords):
            print(f"  [{label}] {filename}")

# checking if there is a wider Drive structure outside of data_storage
PARENT_DRIVE_PATH = '/content/drive/MyDrive/waveform_analysis_outputs'
print(f"\nFull structure under {PARENT_DRIVE_PATH}:")
for root, dirs, files in os.walk(PARENT_DRIVE_PATH):
    for f in files:
        if any(k in f.lower() for k in keywords):
            print(" -", os.path.join(root, f))

In [ ]:
import zipfile
import os

LOCAL_FMA_META = os.path.join(LOCAL_STORAGE, 'fma_metadata')

# unzip the already existing local zip file
with zipfile.ZipFile(os.path.join(LOCAL_STORAGE, 'fma_metadata.zip'), 'r') as zip_ref:
    zip_ref.extractall(LOCAL_STORAGE)

print("Files extracted:")
print(sorted(os.listdir(LOCAL_FMA_META)))

# loading raw_tracks.csv (flat format: track_id, artist_id, ...)
raw_tracks = pd.read_csv(os.path.join(LOCAL_FMA_META, 'raw_tracks.csv'))
print("\nColumns in raw_tracks.csv:")
print(raw_tracks.columns.tolist())

raw_tracks_small = raw_tracks[['track_id', 'artist_id', 'artist_name']].copy()
print("\nSample:")
print(raw_tracks_small.head())

## 1.2. Artist-Level Data Leakage Audit

The `train.csv`, `val.csv`, and `test.csv` files contain only the track ID (`track_id`) and genre (`genre`). This information is not enough to determine whether the same artist appears in more than one dataset split.

To perform this check, the official FMA metadata is used. The file `raw_tracks.csv`, available in `fma_metadata.zip` in the project's data storage, provides the corresponding `artist_id` and `artist_name` for each `track_id`.

The artist information is matched with the three dataset splits to check whether any artist appears in the training, validation, and test sets.

If the same artist appears in both the training and test sets, the model may learn artist-specific characteristics—such as vocal timbre, production style, or mastering style—instead of learning patterns that are generally associated with a particular genre. This could make the reported performance appear higher than it would be when evaluating the model on tracks by completely unseen artists.

In [ ]:
import pandas as pd

# merging 800 records with artist_id by track_id
raw_tracks_small = raw_tracks[['track_id', 'artist_id', 'artist_name']].copy()

audit_with_artists = all_splits_df.merge(raw_tracks_small, on='track_id', how='left')

# check for missing matches (track_id not found in FMA metadata)
missing = audit_with_artists['artist_id'].isna().sum()
print(f"Tracks without matched artist metadata: {missing} / {len(audit_with_artists)}")

# 3. Artist sets по split
train_artists = set(audit_with_artists.loc[audit_with_artists['split'] == 'train', 'artist_id'].dropna())
val_artists = set(audit_with_artists.loc[audit_with_artists['split'] == 'validation', 'artist_id'].dropna())
test_artists = set(audit_with_artists.loc[audit_with_artists['split'] == 'test', 'artist_id'].dropna())

print("\n" + "="*50)
print("ARTIST-LEVEL DATA LEAKAGE AUDIT")
print("="*50)
print(f"Unique artists total:          {audit_with_artists['artist_id'].nunique()}")
print(f"Unique artists in train:       {len(train_artists)}")
print(f"Unique artists in validation:  {len(val_artists)}")
print(f"Unique artists in test:        {len(test_artists)}")
print(f"\nTrain ∩ Validation artists:    {len(train_artists & val_artists)}")
print(f"Train ∩ Test artists:          {len(train_artists & test_artists)}")
print(f"Validation ∩ Test artists:     {len(val_artists & test_artists)}")

# If there is an overlap - how many records (tracks) exactly does it affect?
overlap_train_test = train_artists & test_artists
if overlap_train_test:
    affected = audit_with_artists[
        audit_with_artists['artist_id'].isin(overlap_train_test)
    ]
    print(f"\nTracks affected by Train/Test artist overlap: {len(affected)}")
    print(affected[['split', 'artist_id', 'artist_name', 'genre']].sort_values('artist_id').head(20))

### Interpretation of the Artist-Level Audit

All 800 tracks were successfully matched with the official FMA artist metadata. Since every track has an `artist_id`, the audit covers the entire dataset.

The dataset includes 547 unique artists. However, the same artists appear in multiple splits:

- 49 artists appear in both the training and validation sets.
- 43 artists appear in both the training and test sets.
- 19 artists appear in both the validation and test sets.

In total, 143 tracks are linked to artists who appear in both the training and test sets.

This shows that the original random split does not keep artists fully separate between the different datasets. As a result, the model may encounter artist-specific features during testing that were already present in the training data. These features may include vocal timbre, instrumentation, recording conditions, production style, and mastering choices.

Therefore, the test performance should not be interpreted as a fully artist-independent measure of the model's ability to generalize across music genres. The results may be somewhat optimistic because part of the test set contains artists who also appear in the training set.

This finding does not prove that the model memorized the artists' identities or that the 143 tracks were classified correctly because of the artist overlap. However, it shows that the experimental setup does not prevent the model from relying on this type of shortcut.

For a stricter evaluation, the data should be divided using an artist-disjoint split. In this type of split, no artist appears in more than one of the training, validation, or test sets.

### Stage 1 Conclusion

The dataset contains 800 tracks: 559 in the training set, 121 in the validation set, and 120 in the test set. All selected tracks have complete track-level metadata, and there are no duplicate `track_id` values across the splits.

Using the official FMA metadata, an additional artist-level audit was conducted. This check revealed that the original split is not artist-disjoint: 43 artists appear in both the training and test sets, affecting a total of 143 tracks across these two subsets.

This represents an important methodological limitation. The reported test score reflects performance on a random track split rather than on completely unseen artists. As a result, the model's ability to generalize to new artists may be overestimated.

However, this audit alone does not determine the exact extent of this performance boost. Measuring that impact would require a new experiment with an artist-disjoint split. A logical next step is to prepare an artist-separated split and compare its results against the original random split, without modifying the existing training notebooks.

## 2. Baseline Comparisons & Performance Evaluation

The original experiment does not include any baseline comparisons or clear criteria for judging what counts as a "good" or "better" model. Without this context, a test accuracy of ~33.3% is difficult to interpret on its own.

To better evaluate the model's performance, it is compared against three simple baseline approaches. These baselines use the actual genres from the test set to check the results:

- **Random predictions (`random`)** — Each genre has an equal chance of being selected ($1/K$).
- **Most frequent genre prediction (`most_frequent`)** — Always predicts the genre that appears most often in the training set.
- **Random predictions based on genre distribution (`stratified`)** — The probability of selecting each genre depends on how often it appears in the training set.

The deep learning model is then compared against these baselines using Accuracy, Balanced Accuracy, Macro Precision, Macro Recall, and Macro F1-score, in order to answer the following questions:

- Does the CNN learn genuine musical features, or is its performance mainly driven by random chance or class frequency bias?
- How much value does the deep learning approach actually add compared to these simple strategies?

In [ ]:
# Baseline comparisons and statistical performance evaluation

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score
from sklearn.dummy import DummyClassifier

# еxtract ground truth and trained model predictions from existing test results
# (inference_df contains true_genre and pred_genre / predicted_label)
# check exact column names in inference_df
true_col = [c for c in inference_df.columns if 'true' in c.lower() or 'label' in c.lower() or 'genre' in c.lower()][0]
pred_col = [c for c in inference_df.columns if 'pred' in c.lower()][0]

y_train = train_df['genre'].values
y_test = inference_df[true_col].values
y_pred_cnn = inference_df[pred_col].values

classes = np.unique(y_test)
n_classes = len(classes)

# fit standard Scikit-Learn Dummy Baselines on train set and predict on test set
# setting random_state
dummy_uniform = DummyClassifier(strategy='uniform', random_state=42)
dummy_uniform.fit(train_df[['track_id']], y_train)
y_pred_uniform = dummy_uniform.predict(test_df[['track_id']])

dummy_majority = DummyClassifier(strategy='most_frequent')
dummy_majority.fit(train_df[['track_id']], y_train)
y_pred_majority = dummy_majority.predict(test_df[['track_id']])

dummy_stratified = DummyClassifier(strategy='stratified', random_state=42)
dummy_stratified.fit(train_df[['track_id']], y_train)
y_pred_stratified = dummy_stratified.predict(test_df[['track_id']])

# define evaluation metric computation
def evaluate_predictions(y_true, y_pred, model_name):
    return {
        'Model / Baseline': model_name,
        'Accuracy (%)': round(accuracy_score(y_true, y_pred) * 100, 2),
        'Balanced Acc (%)': round(balanced_accuracy_score(y_true, y_pred) * 100, 2),
        'Macro Precision (%)': round(precision_score(y_true, y_pred, average='macro', zero_division=0) * 100, 2),
        'Macro Recall (%)': round(recall_score(y_true, y_pred, average='macro', zero_division=0) * 100, 2),
        'Macro F1 (%)': round(f1_score(y_true, y_pred, average='macro', zero_division=0) * 100, 2)
    }

# compile comparison table
results = []
results.append(evaluate_predictions(y_test, y_pred_uniform, f'Uniform Random (1/{n_classes})'))
results.append(evaluate_predictions(y_test, y_pred_majority, 'Majority Class (Train Mode)'))
results.append(evaluate_predictions(y_test, y_pred_stratified, 'Stratified Prior-Matching'))
results.append(evaluate_predictions(y_test, y_pred_cnn, 'Trained CNN Model'))

baseline_summary_df = pd.DataFrame(results)

print("=" * 75)
print("BENCHMARK: CNN MODEL VS. HEURISTIC & STATISTICAL BASELINES")
print("=" * 75)
display(baseline_summary_df)

# relative improvement over random guess
cnn_acc = baseline_summary_df.loc[baseline_summary_df['Model / Baseline'] == 'Trained CNN Model', 'Accuracy (%)'].values[0]
random_acc = 100.0 / n_classes
lift_factor = cnn_acc / random_acc

print(f"\nTheoretical Random Baseline Accuracy: {random_acc:.2f}% (1 / {n_classes} classes)")
print(f"CNN Accuracy Lift Factor over Random: {lift_factor:.2f}x")

### Interpretation of the Baseline Comparison

The trained CNN model reached an accuracy of **32.5%**, compared with:

- **12.5%** for random guessing
- **12.5%** for always picking the most common genre
- **15.0%** for guessing based on the frequency of each genre

Since there are eight genres, random guessing has an expected accuracy of around 12.5%. Always picking the most common genre also yields 12.5%, indicating that no single genre is frequent enough in the data to provide high accuracy on its own.

The CNN clearly scores higher than these simple approaches—its accuracy is about **2.6 times** higher than random guessing. This indicates that its predictions contain more useful information than what would be obtained purely by chance or through frequency-based rules.

However, these results do not mean the model performs at a high level overall. The other class-balance metrics are:

- Macro F1-score: **28.61%**
- Macro Precision: **26.25%**
- Macro Recall: **32.50%**

These values show that while overall accuracy is 32.5%, the model's performance varies across the eight genres. Accuracy simply counts the total number of correct predictions, whereas the macro metrics check whether the model performs consistently across all genres, including those it finds challenging.

In summary:

> The CNN performs better than basic guessing strategies, but its overall results remain modest. Scoring higher than simple baselines is a positive sign, but it does not alone demonstrate that the model reliably distinguishes between genres or that deep learning is necessarily better than simpler alternatives.

To gain a clearer understanding of whether the neural network provides real value, it should also be compared with traditional models trained on the same data using precomputed audio features. All models should be evaluated on the same data splits. Additionally, an artist-disjoint split should be used to minimize the chance of the model relying on artist-specific cues.

## 2. Baseline Comparison & Statistical Significance

To verify whether the CNN model performs better than random chance and produces balanced results across all genres, the following evaluation criteria are applied:

1. **Performance must exceed random guessing**

   There are eight music genres in this classification task. If every genre has an equal probability of being selected, the expected accuracy for uniform random guessing is:

   $$\frac{1}{K} = \frac{1}{8} = 0.125 = 12.5\%$$

   Here, $K$ represents the total number of genre classes. The fraction $1/8$ reflects the equal probability assigned to each of the eight possible categories.

   To assess whether the CNN's test accuracy is reliably higher than 12.5%, a 95% Wilson score confidence interval is computed. This interval provides an estimated range for the true accuracy of the model. If the lower bound of this interval remains strictly above 12.5%, it indicates that the model outperforms random guessing with statistical confidence.

   The 95% Wilson score confidence interval formula is:

   $$\frac{\hat{p} + \frac{z^2}{2n} \pm z\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}}{1+\frac{z^2}{n}}$$

   Where:
   - $\hat{p}$ is the observed accuracy on the test set;
   - $n$ is the total number of test samples ($n = 120$);
   - $z$ is the critical value for the chosen confidence level ($z \approx 1.96$ for a 95% confidence interval).

2. **The model must recognize every genre to some extent**

   A model should not be considered acceptable solely based on overall accuracy if it completely fails to recognize one or more classes. Therefore, the minimum recall across all genres must remain strictly positive:

   $$\min_c \text{Recall}_c > 0$$

   Here, $c$ denotes an individual genre class, and $\min_c$ represents the lowest recall among all eight genres. This condition ensures that every genre receives at least one correct prediction.

   The recall for a specific class is defined as:

   $$\text{Recall} = \frac{\text{Correctly predicted tracks for this genre}}{\text{Total actual tracks in this genre}}$$

   This criterion does not require high performance across all genres; it simply confirms that no single genre is entirely ignored by the classifier.

3. **Model superiority requires clear statistical support**

   The notation $M_A \succ M_B$ indicates that model $M_A$ is strictly superior to model $M_B$. This conclusion is drawn only when all of the following conditions are met:

   - $M_A$ achieves a higher Macro F1-score than $M_B$;
   - The performance difference is statistically significant;
   - Both models maintain a non-zero recall across all genre classes ($\min_c \text{Recall}_c > 0$).

   The Macro F1-score is computed as the unweighted arithmetic mean of the per-class F1-scores:

   $$\text{Macro F1} = \frac{1}{K}\sum_{c=1}^{K} \text{F1}_c$$

   This metric gives equal weight to every genre regardless of its support size in the test data.

   The F1-score for an individual genre class is calculated as the harmonic mean of precision and recall:

   $$\text{F1} = 2 \cdot \frac{\text{Precision}\cdot\text{Recall}}{\text{Precision}+\text{Recall}}$$

   This formulation ensures that both false positives (low precision) and false negatives (low recall) are taken into account simultaneously.

Together, these criteria provide a structured framework to determine whether the model outperforms simple baselines and maintains balanced coverage across all target classes.

In [ ]:
# Statistical significance test and Wilson Score 95% Confidence Interval

import numpy as np
import pandas as pd
from scipy import stats

# verify exact 1-to-1 alignment between test_df and inference_df
pred_col = [c for c in inference_df.columns if 'pred' in c.lower()][0]
test_merged = test_df.merge(inference_df[['track_id', pred_col]], on='track_id', how='inner')

y_true = test_merged['genre'].values
y_pred = test_merged[pred_col].values
N = len(y_true)
correct = (y_true == y_pred).sum()
observed_acc = correct / N

# compute 95% Wilson Score Confidence Interval
z = stats.norm.ppf(0.975)  # 1.96
denom = 1 + z**2 / N
p_adj = (observed_acc + z**2 / (2 * N)) / denom
se_adj = z * np.sqrt((observed_acc * (1 - observed_acc) + z**2 / (4 * N)) / N) / denom

ci_lower = max(0.0, p_adj - se_adj)
ci_upper = min(1.0, p_adj + se_adj)

# binomial significance test vs random baseline (12.5%)
p_val_random = stats.binomtest(correct, n=N, p=0.125, alternative='greater').pvalue

print("=" * 60)
print(f"Test Samples (N):         {N} (100% aligned by track_id)")
print(f"Observed Accuracy:        {observed_acc * 100:.2f}% ({correct}/{N} correct)")
print(f"95% Confidence Interval:  [{ci_lower * 100:.2f}% — {ci_upper * 100:.2f}%]")
print(f"Margin of Error:          ±{(ci_upper - ci_lower) / 2 * 100:.2f}%")
print(f"Binomial Test vs Random:  p-value = {p_val_random:.4e}")
print(f"Statistically > 12.5%:    {p_val_random < 0.05}")
print("=" * 60)

### Interpretation of Statistical Significance

The predictions were correctly matched with the test set — all 120 test tracks were found using their `track_id`.

The CNN model correctly classified 39 out of 120 tracks, corresponding to an accuracy of **32.50%**. The 95% Wilson confidence interval for this accuracy is **[24.78%, 41.31%]**. This interval is relatively wide, which is likely due to the limited size of the test set.

The lower bound of the interval — 24.78% — is above the theoretical accuracy of 12.5% expected from uniform random guessing across eight classes. A one-sided binomial test also produced a very low p-value: **9.5625 × 10⁻⁹**. Under the assumptions of this test, this provides statistical support for a difference between the CNN's performance and random guessing.

These results are consistent with the idea that the model's predictions contain information related to the genre labels, rather than being purely random. However, they do not indicate the exact source of this information and do not prove that the model has learned reliable, general musical characteristics.

Statistical significance does not necessarily mean the model is good enough for practical use. The CNN misclassified 81 out of 120 test tracks, and an accuracy of 32.50% remains modest for an eight-class task.

The confidence interval and binomial test assume that the test observations provide sufficiently independent evidence. This assumption may be partially violated, since the artist-level audit showed that some artists appear in both the training and test sets. Tracks by the same artist may share similar acoustic characteristics. As a result, this statistical result applies specifically to the current random track split and cannot, on its own, be considered evidence of good performance on completely unseen artists.

Overall, the results show that the CNN performs better than random guessing under the current data split. However, there is still not enough evidence to conclude that the model is a reliable genre classifier or that its performance is independent of the artists involved. A more complete evaluation would require macro-averaged metrics, per-class recall, comparison with traditional models, and testing on an artist-disjoint dataset split.

## 3. Per-Genre Performance and Analysis of Missed Classes

Overall accuracy, such as the observed value of **32.50%**, provides a general summary of performance across the entire dataset. However, it can hide important differences between individual genres. One important question when evaluating a classifier is whether the model performs reasonably consistently across all classes or completely misses some of them, resulting in:

$$\text{Recall} = 0$$

Recall measures the proportion of actual tracks from a given genre that the model classifies correctly. A value of $\text{Recall} = 0$ means that the model did not correctly identify any of the test tracks belonging to that genre.

This section examines the model's performance for each genre in more detail and considers the following questions:

1. **Are there any genres with zero recall or zero precision?**

   Zero precision means that none of the tracks predicted as a particular genre were actually from that genre. If the model does not predict a genre at all, its precision may also be reported as zero by the evaluation procedure.

   Zero recall means that the model failed to correctly identify any of the actual test tracks belonging to that genre.

2. **What factors could help explain difficulties with particular classes?**

   Several possible explanations should be considered and evaluated rather than assumed:

   - **Limited amount of training data:** The training set contains 559 tracks distributed across 8 classes, which corresponds to an average of approximately 70 tracks per genre:

     $$\frac{559}{8} \approx 69.9$$

     With relatively few examples per class, the neural network may not learn sufficiently reliable patterns for every genre.

   - **Acoustic similarity:** Some genres may use similar instruments, rhythms, or tempos. This can make them difficult to distinguish using spectrogram-based representations alone.

   - **Variation within a genre:** A single genre may include several substyles and different production characteristics. This internal variation can make it more difficult for the model to identify a consistent pattern.

3. **Which genres receive the incorrect predictions?**

   The confusion matrix is used to examine which genres are most often confused with one another. For example, it can show whether tracks from one genre are repeatedly classified as another genre with similar acoustic characteristics.

These analyses help determine whether the model's overall accuracy is supported by relatively balanced performance across genres or is mainly influenced by a subset of classes.

In [ ]:
# Per-class performance analysis and misclassification routing breakdown

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# compute per-class classification report as a structured DataFrame
report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()

# filter only individual genre rows (exclude accuracy, macro avg, weighted avg)
genre_metrics_df = report_df.loc[~report_df.index.isin(['accuracy', 'macro avg', 'weighted avg'])].copy()
genre_metrics_df['support'] = genre_metrics_df['support'].astype(int)
genre_metrics_df[['precision', 'recall', 'f1-score']] = genre_metrics_df[['precision', 'recall', 'f1-score']].round(4) * 100

print("=" * 65)
print("PER-CLASS CLASSIFICATION PERFORMANCE")
print("=" * 65)
display(genre_metrics_df.sort_values(by='recall', ascending=False))

# Identify zero-recall classes
zero_recall_classes = genre_metrics_df[genre_metrics_df['recall'] == 0.0].index.tolist()
print("\n" + "-" * 65)
print(f"Classes with Recall = 0.0%: {zero_recall_classes}")
print("-" * 65)

# analyze misclassification routing (where did true samples go?)
unique_labels = sorted(list(set(y_true) | set(y_pred)))
cm = confusion_matrix(y_true, y_pred, labels=unique_labels)
cm_df = pd.DataFrame(cm, index=[f"True: {l}" for l in unique_labels], columns=[f"Pred: {l}" for l in unique_labels])

print("\n" + "=" * 65)
print("CONFUSION MATRIX: TRUE VS. PREDICTED ROUTING")
print("=" * 65)
display(cm_df)

# detailed destination analysis for zero-recall classes
if zero_recall_classes:
    print("\n" + "=" * 65)
    print("MISCLASSIFICATION DESTINATIONS FOR ZERO-RECALL CLASSES")
    print("=" * 65)
    for zc in zero_recall_classes:
        zc_true_mask = (y_true == zc)
        destinations = pd.Series(y_pred[zc_true_mask]).value_counts()
        print(f"\nTrue Genre '{zc}' (Total {zc_true_mask.sum()} samples) was predicted as:")
        for pred_genre, count in destinations.items():
            print(f"  -> {pred_genre}: {count} tracks ({count / zc_true_mask.sum() * 100:.1f}%)")

### Interpretation of Per-Class Performance and Zero-Recall Cases

The test set contains 15 tracks for each of the eight genres, giving a total of 120 test examples. Since the classes are evenly represented, macro-averaged metrics are particularly useful for this evaluation, as they give equal weight to every genre regardless of how many correct predictions it receives.

The best results are observed for **Hip-Hop**. The model correctly identified 10 of the 15 tracks, corresponding to a recall of 66.67%. Precision was 50.00%, and the F1-score was 57.14%. **Rock** also showed relatively good performance, with a precision of 50.00%, recall of 46.67%, and F1-score of 48.28%.

For **Folk**, recall was 46.67%, while **International** reached 33.33%. **Instrumental** had a recall of 40.00%, but a lower precision of 15.00%. This suggests that while the model correctly identifies some real Instrumental tracks, it also frequently predicts this genre for tracks that actually belong to other classes.

**Experimental** appears to be harder to recognize than some of the other genres. It obtained a recall of 26.67%, precision of 17.39%, and F1-score of 21.05%. These values indicate more uneven performance, but on their own they do not explain why the model struggles with this genre.

The most notable issue concerns the **Electronic** and **Pop** classes. Both report:

- precision = 0.00%
- recall = 0.00%
- F1-score = 0.00%

Each of these classes has 15 examples in the test set, so the zero recall is not caused by a lack of test samples. Instead, it means the model did not correctly identify a single Electronic or Pop track — all tracks from these two genres were assigned to other classes.

According to the confusion matrix, the 15 **Electronic** tracks were classified as follows:

- Folk — 4 tracks
- Experimental — 3
- Instrumental — 3
- Hip-Hop — 2
- International — 2
- Rock — 1

None of them were predicted as Electronic.

The **Pop** tracks were distributed as follows:

- Folk — 5 tracks
- Experimental — 3
- Instrumental — 3
- International — 3
- Rock — 1

Again, none were classified as Pop.

This error pattern suggests that, under the current conditions, the model does not sufficiently distinguish Electronic and Pop from the other genres. However, the results alone do not point to a single specific cause. Possible explanations that would need to be examined through further experiments include:

- acoustic similarity between certain genres;
- high variation within a single genre;
- a limited number of training examples;
- limitations of the spectrogram-based representation;
- artist overlap between the training and test sets.

Overall accuracy should also be interpreted carefully. The model made 39 correct predictions out of 120, corresponding to an accuracy of 32.50%. However, this value does not describe all genres equally well — a substantial share of the correct predictions comes from classes like Hip-Hop and Rock, while Electronic and Pop contribute no correct predictions at all.

The zero-recall cases are therefore an important signal of weakness in the model's current performance. They show that the model fails to recognize two of the eight classes under the data split and experimental setup used here. This is a limitation for balanced multi-class genre classification, even though overall accuracy is higher than what would be expected from random guessing.

Overall, these results are consistent with the idea that the CNN has learned some genre-related patterns. However, they are not sufficient to conclude that the model has a reliable and generally valid representation for all classes. Clarifying the underlying causes would require further analysis, including comparison with traditional models, evaluation of alternative audio representations, and testing on an artist-disjoint data split (an artist-disjoint split means that all tracks from a given artist are placed into only one subset — training, validation, or test — so that artists in the test set are not seen during training).